# atari_50m_endpoint: small memory setting

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import textwrap

import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import lifetime_average, return_curve, subsample
from analysis.stats import mean_ci, min_max_normalize
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each agent's stored runs per game.
AGENTS = {
    "Large recency (1M)": "ddqn_atari",
    "Small recency (20k)": "ddqn_small_atari",
    "Small recency (20k) 10-step": "ddqn_small_nstep_atari",
    "Endpoint replay (20k)": "endpoint_small_atari",
}
component = EXPERIMENT.component("dqn_atari")
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP

results = {}
for agent, name in AGENTS.items():
    for game in GAMES:
        runs = load_results(EXPERIMENT, name, where={"ENV_HYPERS.GAME": game})
        if runs is not None:
            results[agent, game] = runs

In [ ]:
# Compute each agent's mean return per game with a 95% bootstrap CI, and each
# run's step-weighted average return for the bar plot.
curves = {}
scores = {}
for (agent, game), runs in results.items():
    curve = return_curve(runs.reward, runs.done)
    timesteps, returns = subsample(curve)
    curves[agent, game] = (timesteps * FRAMESKIP, mean_ci(returns), len(returns))
    scores[agent, game] = lifetime_average(curve)

In [ ]:
# Style for the per-game learning-curve plots.
GAME_STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 16,
    "legend.frameon": False,
    "legend.loc": "upper left",
}
LINE_WIDTH = 1.75
LINES = {
    "Large recency (1M)": {"color": "black"},
    "Small recency (20k)": {"color": "tab:blue"},
    "Small recency (20k) 10-step": {"color": "tab:purple"},
    "Endpoint replay (20k)": {"color": "tab:red"},
}
BAND_ALPHA = 0.2
LEGEND_LABEL = "{agent} (n={runs})"
XLABEL = "Frames"
YLABEL = "Return"
YLABEL_KW = {"va": "center", "labelpad": 20}
XTICKS = np.array([10, 20, 30, 40, 50]) * 1_000_000
XTICK_LABELS = [f"{tick // 1_000_000}M" for tick in XTICKS]


def title(game):
    return game.replace("_", " ").title()


def draw(ax, agent, game):
    frames, (mean, low, high), runs = curves[agent, game]
    line = {"linewidth": LINE_WIDTH, **LINES[agent]}
    ax.fill_between(frames, low, high, color=line["color"], alpha=BAND_ALPHA)
    label = LEGEND_LABEL.format(agent=agent, runs=runs)
    ax.plot(frames, mean, label=label, **line)

In [ ]:
# Plot one learning curve per game.
game_figures = {}
for game in GAMES:
    drawn = [agent for agent in AGENTS if (agent, game) in curves]
    if not drawn:
        continue
    with plt.rc_context(GAME_STYLE):
        fig, ax = plt.subplots()
        for agent in drawn:
            draw(ax, agent, game)
        ax.set(title=title(game), xlabel=XLABEL)
        ax.set_ylabel(YLABEL, **YLABEL_KW)
        ax.set_xticks(XTICKS, XTICK_LABELS)
        ax.legend()
        fig.tight_layout()
    game_figures[game] = fig
plt.show()

In [ ]:
# Style for the all-games grid, laid out as in atari_50m_deterministic.
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP
GRID_STYLE = {
    "figure.figsize": (16, 9.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (3, 4)
PANELS = {  # (column, row) of each game's panel
    "battle_zone": (0, 0),
    "double_dunk": (0, 1),
    "name_this_game": (0, 2),
    "phoenix": (1, 0),
    "qbert": (1, 1),
    "pong": (1, 2),
    "breakout": (2, 0),
    "seaquest": (2, 1),
    "centipede": (2, 2),
    "ms_pacman": (3, 0),
    "beam_rider": (3, 1),
    "space_invaders": (3, 2),
}
PANEL_ASPECT = 3 / 4
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.06, 1, 1), "w_pad": 0.2, "h_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.055, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {"loc": "upper center", "bbox_to_anchor": (0.5, 0.035), "ncol": 2}

In [ ]:
# Plot every game in one grid. Seed counts can differ per game, so the shared
# legend names only the agents.
grid = None
if curves:
    with plt.rc_context(GRID_STYLE):
        grid, axes = plt.subplots(*GRID_SHAPE)
        legend = {}
        for game, (column, row) in PANELS.items():
            ax = axes[row, column]
            ax.set_box_aspect(PANEL_ASPECT)
            drawn = [agent for agent in AGENTS if (agent, game) in curves]
            if not drawn:
                ax.axis("off")
                continue
            for agent in drawn:
                draw(ax, agent, game)
            ax.set_title(title(game))
            ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
            for agent, handle in zip(drawn, ax.get_lines(), strict=True):
                legend[agent] = handle
        grid.tight_layout(**GRID_LAYOUT)
        grid.supxlabel(**GRID_XLABEL)
        grid.supylabel(**GRID_YLABEL)
        shown = [agent for agent in AGENTS if agent in legend]
        grid.legend([legend[agent] for agent in shown], shown, **GRID_LEGEND)
    plt.show()

In [ ]:
# Style for the normalized-score bar plot.
BAR_ORDER = [
    "Large recency (1M)",
    "Endpoint replay (20k)",
    "Small recency (20k)",
    "Small recency (20k) 10-step",
]
BAR_STYLE = {
    "figure.figsize": (9, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 16,
}
BAR_TITLE = "Min-max normalized step-weighted average return"
BAR_YLABEL = "Normalized return"
BAR_LABEL_WIDTH = 14
ERROR_KW = {"ecolor": "black", "capsize": 6, "elinewidth": 1.5}

In [ ]:
# Normalize each game's runs onto [0, 1] with bounds shared by the four agents,
# average each seed over games, then bootstrap a CI over seeds.
complete = [game for game in GAMES if all((agent, game) in scores for agent in AGENTS)]
normalized = {}
for game in complete:
    stacks = min_max_normalize([scores[agent, game] for agent in BAR_ORDER])
    for agent, stack in zip(BAR_ORDER, stacks, strict=True):
        normalized[agent, game] = stack

bar_figure = None
if complete:
    with plt.rc_context(BAR_STYLE):
        bar_figure, ax = plt.subplots()
        for position, agent in enumerate(BAR_ORDER):
            per_seed = np.mean([normalized[agent, game] for game in complete], axis=0)
            mean, low, high = mean_ci(per_seed)
            error = [[mean - low], [high - mean]]
            color = LINES[agent]["color"]
            ax.bar(position, mean, yerr=error, color=color, error_kw=ERROR_KW)
        labels = [textwrap.fill(agent, BAR_LABEL_WIDTH) for agent in BAR_ORDER]
        ax.set_xticks(range(len(BAR_ORDER)), labels)
        ax.set(title=BAR_TITLE, ylim=(0, 1))
        ax.set_ylabel(BAR_YLABEL)
        bar_figure.tight_layout()
    plt.show()
print(f"bars average {len(complete)} game(s)")

In [ ]:
# Save each game's plot as a PDF in games_small/, and the grid as a PDF and
# PNG in the plots dir, as is the bar plot.
GAMES_DIR = PLOTS_DIR / "games_small"
GAMES_DIR.mkdir(parents=True, exist_ok=True)
for game, fig in game_figures.items():
    fig.savefig(GAMES_DIR / f"{game}.pdf", bbox_inches="tight")
if bar_figure is not None:
    for suffix in ("pdf", "png"):
        bar_figure.savefig(PLOTS_DIR / f"bars_small.{suffix}", bbox_inches="tight")
if grid is not None:
    for suffix in ("pdf", "png"):
        grid.savefig(PLOTS_DIR / f"grid_small.{suffix}", bbox_inches="tight")
print(f"saved {len(game_figures)} game plot(s) to {GAMES_DIR}")